In [10]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import requests

In [ ]:
url = "https://jobinja.ir/jobs?&filters%5Bkeywords%5D%5B0%5D=Data%20specialist&filters%5Bkeywords%5D%5B0%5D=Data%20specialist&preferred_before=1782969301&sort_by=published_at_desc"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://jobinja.ir/",
}

page = requests.get(url, headers=headers)

print(page.status_code)

200


In [3]:
soup = BeautifulSoup(page.text, 'html.parser')

In [4]:
# Our selector requires all five classes but we search using only the stable classes
items = soup.select("li.c-jobListView__item")

print(len(items))

20


In [16]:
# getting name, company, location and type of contract
links = []
company = []
location = []
contract_type = []

for item in items:

    links.append(item.find("a", class_="c-jobListView__titleLink")["href"])

    for meta in item.select("li.c-jobListView__metaItem"):
        icon = meta.find("i")

        if "c-icon--construction" in icon.get("class", []):
            comp = meta.find("span").get_text(strip=True).replace("\u200c", " ")
            company.append(comp)

        elif "c-icon--place" in icon.get("class", []):
            loc = meta.find("span").get_text(strip=True).replace("\u200c", " ")
            location.append(loc)

        elif "c-icon--resume" in icon.get("class", []):
            contract = meta.find("span").find("span").get_text(" ", strip=True)
            contract = " ".join(contract.split()).replace("\u200c", " ")
            contract_type.append(contract)

In [17]:
JobOffers = pd.DataFrame({"likks": links,
                          "company": company,
                          "location": location,
                          "contract_type": contract_type})

In [18]:
JobOffers

,likks,company,location,contract_type
0,https://jobinja.ir/companies/torang-361/jobs/t...,Torang | Torang,تهران، تهران,قرارداد تمام وقت
1,https://jobinja.ir/companies/doondookstudio/jo...,دوندوک استودیو | DoonDookStudio,همدان، همدان,قرارداد تمام وقت
2,https://jobinja.ir/companies/studionamad/jobs/...,استودیو نماد | Studionamad,تهران، تهران,قرارداد تمام وقت
3,https://jobinja.ir/companies/hooman-data/jobs/...,نوین داده هومان | Hooman Data,آذربایجان شرقی، تبریز,قرارداد دورکاری
4,https://jobinja.ir/companies/radman-company/jo...,توسعه تجارت الکترونیک رادمان | Radman Company,تهران، تهران,قرارداد تمام وقت
5,https://jobinja.ir/companies/asa-1/jobs/tGYZ/%...,ویستا سامانه آسا | ASA,تهران، تهران,قرارداد تمام وقت
6,https://jobinja.ir/companies/asa-1/jobs/tGYk/%...,ویستا سامانه آسا | ASA,تهران، تهران,قرارداد تمام وقت
7,https://jobinja.ir/companies/rahkar-fanavari-r...,راهکار فناوری رویان | Rahkar Fanavari Royan,تهران، تهران,قرارداد تمام وقت
8,https://jobinja.ir/companies/naghsh-aval-keyfi...,نقش اول کیفیت | Naghsh Aval Keyfiat,تهران، تهران,قرارداد تمام وقت
9,https://jobinja.ir/companies/qbit/jobs/taBs/%D...,تومن | Toman,تهران، تهران,قرارداد تمام وقت


In [19]:
columns = [
    "definition",
    "experience",
    "salary",
    "sex",
    "military_status",
    "degree_requir",
    "skills",
]

JobOffers[columns] = None

In [21]:
headers_post = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://jobinja.ir/",
}

for i in range(len(JobOffers)):
    url_post = str(JobOffers["likks"][i])

    try:
        page_post = requests.get(url_post, headers=headers_post, timeout=15)
    except requests.RequestException:
        continue

    if page_post.status_code != 200:
        continue
    else:

        soup_post = BeautifulSoup(page_post.text, 'html.parser')

        # JOB_DEFINITINN
        job_definition = soup_post.find("div", class_="o-box__text")
        if job_definition:
            JobOffers.at[i, "definition"] = job_definition.get_text(strip=True)

        # EXPERIENCE & SALARY
        up_btn = soup_post.find("ul", class_="c-jobView__firstInfoBox c-infoBox")
        if up_btn:   
            for li in up_btn.find_all("li", class_="c-infoBox__item"):
                title = li.find("h4", class_="c-infoBox__itemTitle").get_text(strip=True)

                if title == "حداقل سابقه کار":
                    experience = li.find("span", class_="black").get_text(strip=True)
                    JobOffers.at[i, "experience"] = experience

                elif title == "حقوق":
                    salary = li.find("span", class_="black").get_text(strip=True)
                    JobOffers.at[i, "salary"] = salary

        # GENDER & MILITARY_STATUS & DEGREEE & SKILLS
        down_btn = soup_post.find("ul", class_="c-infoBox u-mB0")
        if down_btn:
            for li in down_btn.find_all("li", class_="c-infoBox__item"):
                title = li.find("h4", class_="c-infoBox__itemTitle").get_text(strip=True)

                if title == "جنسیت":
                    sex = li.find("span", class_="black").get_text(strip=True).replace("\u200c", " ")
                    JobOffers.at[i, "sex"] = sex

                elif title == "وضعیت نظام وظیفه":
                    military_status = li.find("span", class_="black").get_text(strip=True).replace("\u200c", "")
                    JobOffers.at[i, "military_status"] = military_status

                elif title == "حداقل مدرک تحصیلی":
                    degree_requir = li.find("span", class_="black").get_text(strip=True).replace("\u200c", " ")
                    JobOffers.at[i, "degree_requir"] = degree_requir

                elif title == "مهارت‌های مورد نیاز":
                    skills = [span.get_text(strip=True) for span in li.find_all("span", class_="black")]
                    JobOffers.at[i, "skills"] = skills